# Ad-Streams Propensity Model — Experiment & Register

Interactive model development for the Ad-Streams propensity engine.

This notebook:
1. Loads engagement features from `dt_user_features` (leakage-free)
2. Races **4 model families** (Logistic Regression, Random Forest, XGBoost, LightGBM) as tracked **ML Experiment** runs
3. Compares them by ROC-AUC and selects the champion
4. Registers the winner to the **Model Registry** and sets the `PROD` pointer

The registered model is then called by `dt_user_propensity` in the streaming pipeline.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

session = get_active_session()
session.use_schema("DEMO_ATAHIR.AD_STREAMS")
print("Account:", session.get_current_account())

## Load features and build the label

**Label:** `CONVERTED = 1` if the user has any conversion.

**Features (10, engagement-only):** click / impression / paid-search counts over 1h, 24h, 7d windows plus 24h event velocity. We deliberately exclude `conversions_total`, `last_conversion_ts`, and `time_since_last_conversion_hrs` to avoid leaking the label.

In [ ]:
FEATURES = [
    "CLICKS_1H", "CLICKS_24H", "CLICKS_7D",
    "IMPRESSIONS_1H", "IMPRESSIONS_24H", "IMPRESSIONS_7D",
    "PAID_SEARCH_1H", "PAID_SEARCH_24H", "PAID_SEARCH_7D",
    "EVENT_VELOCITY_24H",
]

df = session.sql(f"""
    SELECT {', '.join(FEATURES)},
           IFF(conversions_total > 0, 1, 0) AS CONVERTED
    FROM DEMO_ATAHIR.AD_STREAMS.dt_user_features
""").to_pandas()

X = df[FEATURES].astype(float)
y = df["CONVERTED"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Positive rate: {y.mean():.2f}")
X_train.head()

## Experiment: race 4 model families

Each model is a tracked **run** in `AD_PROPENSITY_EXPERIMENT`. Params and metrics are logged so we can compare them side by side in Snowsight (AI & ML » Experiments).

In [ ]:
exp = ExperimentTracking(
    session, database_name="DEMO_ATAHIR", schema_name="ML_EXPERIMENTS"
)
exp.set_experiment("AD_PROPENSITY_EXPERIMENT")

candidates = {
    "logreg":        LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "xgboost":       XGBClassifier(n_estimators=200, max_depth=4, eval_metric="logloss"),
    "lightgbm":      LGBMClassifier(n_estimators=200, max_depth=4, verbose=-1),
}

results = {}
for name, model in candidates.items():
    with exp.start_run(name):
        model.fit(X_train, y_train)
        proba = model.predict_proba(X_test)[:, 1]
        preds = model.predict(X_test)
        auc = roc_auc_score(y_test, proba)
        acc = accuracy_score(y_test, preds)
        f1  = f1_score(y_test, preds, zero_division=0)
        exp.log_params({k: str(v) for k, v in model.get_params().items()})
        exp.log_metrics({"roc_auc": auc, "accuracy": acc, "f1": f1})
        results[name] = {"model": model, "roc_auc": auc, "accuracy": acc, "f1": f1}
        print(f"{name:14s} AUC={auc:.3f}  ACC={acc:.3f}  F1={f1:.3f}")

In [ ]:
# Compare and pick the champion
leaderboard = pd.DataFrame(
    [{"model": k, **{m: v[m] for m in ("roc_auc", "accuracy", "f1")}} for k, v in results.items()]
).sort_values("roc_auc", ascending=False).reset_index(drop=True)
champion_name = leaderboard.iloc[0]["model"]
champion = results[champion_name]["model"]
print(f"Champion: {champion_name}")
leaderboard

## Register the champion to the Model Registry

The model logs as an **IMMUTABLE** function, so it can be called inside the **incremental** `dt_user_propensity` Dynamic Table without forcing a full refresh. Promoting a new version later is a one-line `default` flip — no pipeline DDL change.

In [ ]:
reg = Registry(session, database_name="DEMO_ATAHIR", schema_name="ML_REGISTRY")

mv = reg.log_model(
    champion,
    model_name="AD_PROPENSITY_MODEL",
    version_name="V1",
    sample_input_data=X_train,
    conda_dependencies=["scikit-learn", "xgboost", "lightgbm"],
    metrics={
        "roc_auc": float(results[champion_name]["roc_auc"]),
        "accuracy": float(results[champion_name]["accuracy"]),
        "f1": float(results[champion_name]["f1"]),
        "champion": champion_name,
    },
    comment=f"Propensity model. Champion={champion_name}. Selected by ROC-AUC across 4 families.",
    task=task.Task.TABULAR_BINARY_CLASSIFICATION,
)
print("Registered:", mv.model_name, mv.version_name)
print("Functions:", [f["name"] for f in mv.show_functions()])

# Promote V1 to the PROD pointer (default version)
m = reg.get_model("AD_PROPENSITY_MODEL")
m.default = "V1"
print("Default version:", m.default.version_name)

# Smoke-test inference; note the positive-class output column the DT will read
preds = mv.run(X_test.head(5), function_name="predict_proba")
print(preds.columns.tolist())
preds